# 04 — Mô hình 2: Pose + LSTM/GRU

Nhóm 05 — Fall Detection. Bật **GPU**. Thêm dataset URFD & Le2i qua **Add Input**.


**Cài thư viện** (Kaggle/Colab) — bỏ qua nếu môi trường đã có sẵn.

In [ ]:
!pip -q install ultralytics lap

## hư viện mã nguồn (nhúng sẵn trong notebook)
Toàn bộ hàm/lớp cần dùng được định nghĩa trong các cell dưới đây. Mỗi cell là một nhóm chức năng kèm mô tả; **chạy lần lượt từ trên xuống một lần** trước khi sang phần thực nghiệm.

**Tiện ích chung (seed, I/O, JSON)**

Cố định seed để tái lập (`set_seed`), kiểm tra môi trường Kaggle (`on_kaggle`) và các hàm I/O JSON.

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Tiện ích chung: seed tái lập, nạp config, I/O cơ bản."""
import os
import json
import random
from pathlib import Path

import numpy as np


def set_seed(seed: int = 42) -> None:
    """Cố định seed để kết quả tái lập được (yêu cầu nộp bài của thầy)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def load_config(path: str | Path) -> dict:
    """Nạp file YAML cấu hình."""
    import yaml
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def save_json(obj, path: str | Path) -> None:
    ensure_dir(Path(path).parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path: str | Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def on_kaggle() -> bool:
    return os.path.exists("/kaggle/input")


**Đặc trưng hình học / pose + giảm chiều (PCA/t-SNE)**

Trích **đặc trưng hình học** từ khung bao (`bbox_geometric_features`), **keypoint pose** (`pose_keypoints`), **embedding CNN đóng băng** (`cnn_embeddings`) và hàm **giảm chiều PCA/t-SNE** vẽ scatter (`reduce_and_plot`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Trích đặc trưng + giảm chiều + trực quan hóa (đáp ứng §2.2 đề cương).

- Đặc trưng hình học từ khung bao (cho M1 và phân tích).
- Đặc trưng keypoint pose (cho M2).
- Embedding CNN đóng băng (cho M3).
- PCA / t-SNE để giảm chiều + vẽ scatter tô màu theo nhãn (BẮT BUỘC).
"""
import numpy as np


def bbox_geometric_features(track_seq) -> np.ndarray:
    """Từ chuỗi khung bao (list[Track]) -> đặc trưng hình học theo thời gian.

    Trả vector gồm: tỷ lệ w/h, vận tốc tâm theo phương dọc, gia tốc, mức bất động.
    Dùng cho Mô hình 1 (rule-based) và cho scatter PCA/t-SNE.
    """
    if len(track_seq) < 2:
        return np.zeros(5, dtype=np.float32)
    cx = np.array([t.x + t.w / 2 for t in track_seq])
    cy = np.array([t.y + t.h / 2 for t in track_seq])
    ar = np.array([t.w / max(t.h, 1e-6) for t in track_seq])     # aspect ratio
    vy = np.diff(cy)                                             # vận tốc dọc
    ay = np.diff(vy) if len(vy) > 1 else np.array([0.0])          # gia tốc dọc
    speed = np.sqrt(np.diff(cx) ** 2 + np.diff(cy) ** 2)
    stillness = float((speed[-3:] < speed.mean() * 0.3).mean()) if len(speed) >= 3 else 0.0
    return np.array([ar.max(), ar[-1], vy.max(), ay.max(), stillness], dtype=np.float32)


def normalize_keypoints(kp_xy: np.ndarray, box) -> np.ndarray:
    """Chuẩn hóa keypoint về [0,1] theo khung bao -> bất biến vị trí & kích thước.

    kp_xy: [K,2] toạ độ pixel; box: (x, y, w, h). Trả [K,2].
    """
    x, y, w, h = box
    out = kp_xy.astype(np.float32).copy()
    out[:, 0] = (out[:, 0] - x) / max(w, 1e-6)
    out[:, 1] = (out[:, 1] - y) / max(h, 1e-6)
    return np.clip(out, -0.5, 1.5)


def pose_keypoints(video_source, model_name: str = "yolo11n-pose.pt",
                   conf: float = 0.25, tracker: str = "bytetrack.yaml"):
    """Trích keypoint theo ID người bằng YOLO11-pose (có tracking).

    Trả về dict[track_id] -> np.ndarray[T, K, 2] đã chuẩn hóa theo khung bao.
    """
    from ultralytics import YOLO
    model = YOLO(model_name)
    by_id: dict[int, list] = {}
    results = model.track(source=video_source, persist=True, tracker=tracker,
                          classes=[0], conf=conf, stream=True, verbose=False)
    for r in results:
        if r.boxes is None or r.boxes.id is None or r.keypoints is None:
            continue
        ids = r.boxes.id.int().cpu().numpy()
        boxes = r.boxes.xywh.cpu().numpy()
        kps = r.keypoints.xy.cpu().numpy()          # [n, K, 2]
        for tid, box, kp in zip(ids, boxes, kps):
            cx, cy, w, h = box
            norm = normalize_keypoints(kp, (cx - w / 2, cy - h / 2, w, h))
            by_id.setdefault(int(tid), []).append(norm)
    return {k: np.stack(v) for k, v in by_id.items() if len(v) > 0}


def _build_backbone(backbone: str):
    import torch
    from torchvision import models
    if backbone == "mobilenet_v2":
        net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        feat = torch.nn.Sequential(net.features, torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten())
    else:
        net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        feat = torch.nn.Sequential(*list(net.children())[:-1], torch.nn.Flatten())
    feat.eval()
    for p in feat.parameters():
        p.requires_grad = False
    return feat


def cnn_embeddings(crops, backbone: str = "mobilenet_v2", device: str | None = None):
    """Backbone CNN pretrained ĐÓNG BĂNG -> đặc trưng mỗi frame.

    crops: np.ndarray[T,3,H,W] đã chuẩn hóa ImageNet. Trả np.ndarray[T, D].
    """
    import torch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    feat = _build_backbone(backbone).to(device)
    x = torch.as_tensor(np.asarray(crops), dtype=torch.float32, device=device)
    with torch.no_grad():
        emb = feat(x)
    return emb.cpu().numpy()


def reduce_and_plot(X: np.ndarray, y: np.ndarray, method: str = "tsne", out_png: str = "outputs/embedding.png"):
    """Giảm chiều (PCA/t-SNE) và vẽ scatter 2D tô màu theo nhãn fall/non-fall."""
    from pathlib import Path
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE

    if method == "pca":
        Z = PCA(n_components=2, random_state=42).fit_transform(X)
    else:
        Z = TSNE(n_components=2, random_state=42, perplexity=min(30, max(5, len(X) // 4))).fit_transform(X)

    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(6, 5))
    for lab, name, c in [(0, "Bình thường", "#1f77b4"), (1, "Té ngã", "#d62728")]:
        m = y == lab
        plt.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.6, label=name, color=c)
    plt.legend(); plt.title(f"{method.upper()} đặc trưng (fall vs non-fall)")
    plt.tight_layout(); plt.savefig(out_png, dpi=130); plt.close()
    return out_png


**Độ đo đánh giá mức sự kiện (±T giây)**

Độ đo **mức sự kiện** dùng chung cho M1/M2/M3: so khớp cảnh báo với sự kiện ngã trong cửa sổ dung sai ±T giây (`evaluate_events`), gộp cụm cảnh báo (`merge_alarms`), gộp toàn tập (`aggregate_over_dataset`) và xuất bảng kết quả CSV (`results_table`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Độ đo đánh giá THỐNG NHẤT cho cả 3 mô hình.

Trọng tâm: mức SỰ KIỆN với cửa sổ dung sai ±T giây (định nghĩa chốt trong kế hoạch).
Một cảnh báo là TP nếu rơi trong [fall_start - T, fall_end + T]; ngoài ra là FP;
sự kiện ngã không có cảnh báo nào trùng = FN.
"""
from dataclasses import dataclass


@dataclass
class EventMetrics:
    precision: float
    recall: float
    f1: float
    false_alarm_per_hour: float
    mean_latency_sec: float
    tp: int
    fp: int
    fn: int


def evaluate_events(pred_alarms, gt_falls, video_seconds: float,
                    fps: float, tolerance_sec: float = 1.0) -> EventMetrics:
    """So khớp cảnh báo dự đoán với sự kiện ngã thật ở mức sự kiện.

    pred_alarms: list[frame_index] các frame phát cảnh báo (đã gộp cụm liền kề).
    gt_falls:    list[(start_frame, end_frame)] các sự kiện ngã thật.
    """
    tol = tolerance_sec * fps
    matched_gt = set()
    tp = 0
    latencies = []
    for a in sorted(pred_alarms):
        hit = None
        for gi, (s, e) in enumerate(gt_falls):
            if gi in matched_gt:
                continue
            if s - tol <= a <= e + tol:
                hit = gi
                latencies.append(max(0.0, (a - s) / fps))
                break
        if hit is not None:
            matched_gt.add(hit)
            tp += 1
    fp = len(pred_alarms) - tp
    fn = len(gt_falls) - len(matched_gt)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    far = fp / (video_seconds / 3600) if video_seconds else 0.0
    lat = sum(latencies) / len(latencies) if latencies else 0.0
    return EventMetrics(precision, recall, f1, far, lat, tp, fp, fn)


def merge_alarms(flags, min_gap: int = 5):
    """Gộp các frame cảnh báo liên tiếp thành 1 sự kiện (lấy frame đầu mỗi cụm)."""
    frames = [i for i, v in enumerate(flags) if v]
    if not frames:
        return []
    events, prev = [frames[0]], frames[0]
    for f in frames[1:]:
        if f - prev > min_gap:
            events.append(f)
        prev = f
    return events


def aggregate_over_dataset(per_video: list[EventMetrics], video_seconds_total: float) -> EventMetrics:
    """Gộp EventMetrics trên nhiều video (micro: cộng dồn tp/fp/fn)."""
    tp = sum(m.tp for m in per_video)
    fp = sum(m.fp for m in per_video)
    fn = sum(m.fn for m in per_video)
    lat = [m.mean_latency_sec for m in per_video if m.tp > 0]
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    far = fp / (video_seconds_total / 3600) if video_seconds_total else 0.0
    mean_lat = sum(lat) / len(lat) if lat else 0.0
    return EventMetrics(precision, recall, f1, far, mean_lat, tp, fp, fn)


def results_table(rows: list[dict], out_csv: str | None = None):
    """Tạo bảng so sánh (model x protocol) từ các dòng kết quả; xuất CSV cho báo cáo.

    rows: [{'model':..., 'protocol':..., 'precision':..., 'recall':..., 'f1':...,
            'false_alarm_per_hour':..., 'mean_latency_sec':..., 'fps':...}, ...]
    """
    import pandas as pd
    df = pd.DataFrame(rows)
    if out_csv:
        from pathlib import Path
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return df


**YOLO11 + ByteTrack (phát hiện + bám vết người)**

`Track` + `DetectorTracker`: chạy **YOLO11 phát hiện người + ByteTrack bám ID** trên một video; `write_mot` xuất kết quả chuẩn MOT.

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Pipeline dùng chung: YOLO11 (phát hiện người) + ByteTrack (gán ID).

Dùng tracker tích hợp của Ultralytics (model.track(..., tracker="bytetrack.yaml")).
Xuất kết quả chuẩn MOT để tính IDF1 và cho các mô hình phân loại dùng lại.
"""
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Track:
    frame: int
    track_id: int
    x: float
    y: float
    w: float
    h: float
    conf: float


class DetectorTracker:
    def __init__(self, model_name: str = "yolo11n.pt", conf: float = 0.25, person_class: int = 0):
        from ultralytics import YOLO
        self.model = YOLO(model_name)
        self.conf = conf
        self.person_class = person_class

    def run(self, source, tracker: str = "bytetrack.yaml"):
        """Chạy detection + tracking trên 1 video, trả về list[Track].

        source: đường dẫn video (Le2i) hoặc list ảnh/thư mục (URFD).
        """
        tracks: list[Track] = []
        results = self.model.track(
            source=source, persist=True, tracker=tracker,
            classes=[self.person_class], conf=self.conf, stream=True, verbose=False,
        )
        for fi, r in enumerate(results):
            if r.boxes is None or r.boxes.id is None:
                continue
            for box, tid, cf in zip(r.boxes.xywh.cpu().numpy(),
                                    r.boxes.id.int().cpu().numpy(),
                                    r.boxes.conf.cpu().numpy()):
                x, y, w, h = box  # tâm + kích thước
                tracks.append(Track(fi, int(tid), float(x - w / 2), float(y - h / 2),
                                    float(w), float(h), float(cf)))
        return tracks


def write_mot(tracks: list[Track], out_path: str | Path) -> None:
    """Ghi file chuẩn MOT: frame,id,x,y,w,h,conf,-1,-1,-1 (phục vụ TrackEval)."""
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for t in tracks:
            f.write(f"{t.frame + 1},{t.track_id},{t.x:.2f},{t.y:.2f},"
                    f"{t.w:.2f},{t.h:.2f},{t.conf:.3f},-1,-1,-1\n")


**Sinh mẫu & Dataset (cửa sổ trượt, ground-truth)**

Sinh ground-truth sự kiện ngã (`make_gt_falls`), cắt **cửa sổ trượt đặc trưng hình học** (`geometric_windows`) và các `Dataset` torch + `pad_collate` cho M2/M3.

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Xây mẫu huấn luyện/đánh giá từ video -> tracks -> chuỗi cửa sổ trượt.

- make_gt_falls: sinh danh sách sự kiện ngã thật (frame) cho một video.
- geometric_windows: cửa sổ đặc trưng hình học (numpy, KHÔNG cần torch) — dùng
  cho rule-based/phân tích và test nhanh.
- PoseSequenceDataset / CropSequenceDataset: Dataset torch cho M2 / M3.

Lưu ý: hàm chạy detector/pose cần cv2 + ultralytics (có trên Kaggle).
"""
from pathlib import Path

import numpy as np



def _field(item, name):
    """Đọc thuộc tính từ VideoItem hoặc dict (splits['_items'])."""
    return item.get(name) if isinstance(item, dict) else getattr(item, name)


def video_fps(item) -> float:
    """fps gốc của video (URFD ~30, Le2i ~25)."""
    return 25.0 if _field(item, "dataset") == "le2i" else 30.0


def make_gt_falls(item, n_frames: int) -> list[tuple[int, int]]:
    """Danh sách (start, end) sự kiện ngã thật cho một video.

    - Le2i: dùng fall_start/fall_end có trong annotation.
    - URFD: không có mốc frame -> coi cả chuỗi fall là một sự kiện ngã
      (chuỗi URFD ngắn, mỗi chuỗi chứa đúng một lần ngã).
    - Video ADL (label 0): không có sự kiện ngã.
    """
    label = _field(item, "label")
    if label == 0:
        return []
    fs = _field(item, "fall_start")
    fe = _field(item, "fall_end")
    if fs is not None and fe is not None:
        return [(int(fs), int(fe))]
    return [(0, max(1, n_frames - 1))]


def _tracks_by_id(tracks):
    """Gom list[Track] theo track_id, sắp theo frame."""
    by_id: dict[int, list] = {}
    for t in tracks:
        by_id.setdefault(t.track_id, []).append(t)
    for k in by_id:
        by_id[k].sort(key=lambda t: t.frame)
    return by_id


def _label_window(frames: list[int], gt_falls: list[tuple[int, int]], tol: int = 0) -> int:
    """Nhãn 1 nếu cửa sổ frame chồng lấn khoảng ngã thật."""
    for s, e in gt_falls:
        if frames[-1] >= s - tol and frames[0] <= e + tol:
            return 1
    return 0


def geometric_windows(tracks, gt_falls, seq_len: int = 16, stride: int = 8):
    """Cửa sổ trượt đặc trưng hình học -> (X[N,5], y[N]). Không cần torch."""
    X, y = [], []
    for _tid, seq in _tracks_by_id(tracks).items():
        for i in range(0, max(1, len(seq) - seq_len + 1), stride):
            win = seq[i:i + seq_len]
            if len(win) < 3:
                continue
            X.append(bbox_geometric_features(win))
            frames = [t.frame for t in win]
            y.append(_label_window(frames, gt_falls))
    if not X:
        return np.zeros((0, 5), np.float32), np.zeros((0,), np.int64)
    return np.stack(X), np.array(y, np.int64)


# ----------------------- Torch datasets (M2, M3) -----------------------
try:
    import torch
    from torch.utils.data import Dataset

    class PoseSequenceDataset(Dataset):
        """Chuỗi keypoint [T, K*2] -> nhãn. samples: list[(np.ndarray[T,K*2], int)]."""
        def __init__(self, samples):
            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, i):
            x, lab = self.samples[i]
            return torch.tensor(x, dtype=torch.float32), torch.tensor(lab, dtype=torch.float32)

    class CropSequenceDataset(Dataset):
        """Chuỗi ảnh vùng người [T,3,H,W] -> nhãn. samples: list[(np.ndarray[T,3,H,W], int)]."""
        def __init__(self, samples):
            self.samples = samples

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, i):
            x, lab = self.samples[i]
            return torch.tensor(x, dtype=torch.float32), torch.tensor(lab, dtype=torch.float32)

    def pad_collate(batch):
        """Gộp batch chuỗi khác độ dài bằng pad về max T."""
        xs, ys = zip(*batch)
        T = max(x.shape[0] for x in xs)
        xs_p = []
        for x in xs:
            if x.shape[0] < T:
                pad = torch.zeros((T - x.shape[0], *x.shape[1:]), dtype=x.dtype)
                x = torch.cat([x, pad], 0)
            xs_p.append(x)
        return torch.stack(xs_p), torch.stack(ys)
except ImportError:
    PoseSequenceDataset = CropSequenceDataset = None

    def pad_collate(batch):  # type: ignore
        raise ImportError("Cần torch cho pad_collate.")


**Mô hình 2 — Pose + LSTM/GRU**

Mô hình 2 — **Pose + LSTM/GRU** (`PoseLSTM`) và hàm `train` (ủy quyền cho vòng huấn luyện `fit`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Mô hình 2 — Skeleton + chuỗi thời gian (keypoint -> LSTM/GRU).

Đầu vào: chuỗi keypoint đã chuẩn hóa [T, K*2]. Đầu ra: xác suất té ngã.
Nhẹ, ít tham số, phù hợp dữ liệu nhỏ.
"""

try:
    import torch
    import torch.nn as nn

    class PoseLSTM(nn.Module):
        def __init__(self, in_dim: int = 34, hidden: int = 128, layers: int = 1,
                     dropout: float = 0.3, rnn: str = "lstm"):
            super().__init__()
            RNN = nn.LSTM if rnn == "lstm" else nn.GRU
            self.rnn = RNN(in_dim, hidden, layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, 1))

        def forward(self, x):            # x: [B, T, in_dim]
            out, _ = self.rnn(x)
            return self.head(out[:, -1]).squeeze(-1)   # logit [B]
except ImportError:
    PoseLSTM = None  # torch chưa cài (ví dụ môi trường chỉ dựng kế hoạch)


def train(model, loader_train, loader_val, cfg, ckpt_path: str = "outputs/model2_best.pt"):
    """Huấn luyện M2 (ủy quyền cho trainer.fit). Lưu best -> ckpt_path."""
    return fit(model, loader_train, loader_val, cfg, ckpt_path)


**Vòng huấn luyện chung (fit, early-stopping)**

Vòng huấn luyện dùng chung (`fit`): BCEWithLogitsLoss có trọng số lớp, tối ưu F1 trên val, early-stopping và lưu best checkpoint.

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Vòng huấn luyện dùng chung cho M2 (Pose+LSTM) và M3 (CNN+GRU).

Tối ưu F1 mức cửa sổ trên tập val (proxy cho F1 sự kiện), BCEWithLogitsLoss,
Adam chỉ cập nhật tham số requires_grad=True (backbone M3 đã đóng băng),
early-stopping + lưu checkpoint tốt nhất. Cố định seed trước khi gọi.
"""
from pathlib import Path

import numpy as np


def _f1_from_logits(logits, targets, thresh: float = 0.5):
    import torch
    prob = torch.sigmoid(logits)
    pred = (prob >= thresh).int()
    tgt = targets.int()
    tp = int(((pred == 1) & (tgt == 1)).sum())
    fp = int(((pred == 1) & (tgt == 0)).sum())
    fn = int(((pred == 0) & (tgt == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0


def fit(model, loader_train, loader_val, cfg: dict, ckpt_path: str,
        device: str | None = None):
    """Huấn luyện và lưu best checkpoint. Trả về (best_f1, lịch sử).

    cfg cần: lr, epochs; (tùy chọn) patience, pos_weight.
    """
    import torch
    import torch.nn as nn

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=cfg.get("lr", 1e-3))
    pos_w = torch.tensor([cfg.get("pos_weight", 3.0)], device=device)   # lớp té ngã hiếm
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    epochs = cfg.get("epochs", 30)
    patience = cfg.get("patience", 8)

    best_f1, best_state, wait, hist = -1.0, None, 0, []
    for ep in range(epochs):
        model.train()
        tr_loss = 0.0
        for xb, yb in loader_train:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logit = model(xb)
            loss = criterion(logit, yb)
            loss.backward()
            opt.step()
            tr_loss += float(loss) * len(yb)
        tr_loss /= max(1, len(loader_train.dataset))

        # validate
        model.eval()
        logits_all, tgts_all = [], []
        with torch.no_grad():
            for xb, yb in loader_val:
                logits_all.append(model(xb.to(device)).cpu())
                tgts_all.append(yb)
        if logits_all:
            import torch as T
            f1 = _f1_from_logits(T.cat(logits_all), T.cat(tgts_all))
        else:
            f1 = 0.0
        hist.append({"epoch": ep, "train_loss": tr_loss, "val_f1": f1})
        print(f"[ep {ep:02d}] train_loss={tr_loss:.4f}  val_f1={f1:.4f}")

        if f1 > best_f1:
            best_f1, best_state, wait = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stop tại epoch {ep} (best val_f1={best_f1:.4f}).")
                break

    if best_state is not None:
        Path(ckpt_path).parent.mkdir(parents=True, exist_ok=True)
        import torch as T
        T.save(best_state, ckpt_path)
        model.load_state_dict(best_state)
        print(f"Đã lưu best checkpoint -> {ckpt_path} (val_f1={best_f1:.4f})")
    return best_f1, hist


**Đọc frame video + augmentation**

Đọc frame video (hỗ trợ cả thư mục PNG của URFD lẫn file .avi của Le2i) qua `read_frames`, và augmentation nhẹ (`build_augment`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Đọc video, trích frame (lấy mẫu theo fps) và tăng cường dữ liệu."""
from pathlib import Path
from typing import Iterator

import numpy as np


def read_frames(video_path: str | Path, fps_sample: int | None = None) -> Iterator[tuple[int, np.ndarray]]:
    """Duyệt frame của video. URFD là chuỗi PNG, Le2i là .avi -> xử lý cả hai.

    Yields (frame_index, frame_bgr).
    """
    import cv2
    p = Path(video_path)
    if p.is_dir():  # URFD: thư mục chứa PNG
        frames = sorted(p.glob("*.png"))
        step = 1
        if fps_sample:  # URFD ~30fps
            step = max(1, round(30 / fps_sample))
        for i, fp in enumerate(frames):
            if i % step == 0:
                yield i, cv2.imread(str(fp))
    else:  # Le2i: file video
        cap = cv2.VideoCapture(str(p))
        src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
        step = max(1, round(src_fps / fps_sample)) if fps_sample else 1
        idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if idx % step == 0:
                yield idx, frame
            idx += 1
        cap.release()


def build_augment(train: bool = True):
    """Augmentation nhẹ cho dữ liệu nhỏ: lật ngang, đổi sáng, mờ, cắt xén.

    Trả về callable(img)->img. Dùng albumentations nếu có, fallback OpenCV.
    """
    try:
        import albumentations as A
        if not train:
            return A.Compose([A.Resize(224, 224)])
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.MotionBlur(blur_limit=5, p=0.2),
            A.RandomResizedCrop(224, 224, scale=(0.8, 1.0), p=0.5),
            A.Resize(224, 224),
        ])
    except ImportError:
        import cv2
        def _aug(img):
            return cv2.resize(img, (224, 224))
        return _aug


# Gợi ý augmentation theo thời gian (video-level): đổi tốc độ phát (±), bỏ frame.
# TODO (TV1): hàm time_augment(sequence) cho M2/M3.


**Pipeline end-to-end (video → cảnh báo → độ đo)**

Gắn kết end-to-end: trích crop theo ID (`extract_id_crops`), suy luận cảnh báo cho từng mô hình (`alarms_rule`, `alarms_sequence_model`), chạy M1 trên cả tập (`run_rule_on_testset`), dựng mẫu huấn luyện M2/M3 (`build_pose_samples`, `build_crop_embedding_samples`) và bọc model thành hàm dự đoán (`make_predict_fn`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Glue end-to-end: video -> detect/track -> (M1/M2/M3) -> cảnh báo -> EventMetrics.

Kết nối detection_tracking + features + models + evaluate. Dùng cho notebook 06
(đánh giá thống nhất) và cho Web Demo. Phần đọc ảnh/pose cần cv2 + ultralytics.
"""
from pathlib import Path

import numpy as np


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


# ----------------------- trích crop vùng người theo ID -----------------------
def extract_id_crops(video_source, tracks, size: int = 224):
    """Trả dict[track_id] -> (frames[list[int]], crops np.ndarray[T,3,size,size]).

    Đọc lại video theo frame, cắt vùng bbox của từng ID, resize + chuẩn hóa ImageNet.
    """
    import cv2

    by_id = _tracks_by_id(tracks)
    # map frame -> list[(tid, box)]
    frame_boxes: dict[int, list] = {}
    for tid, seq in by_id.items():
        for t in seq:
            frame_boxes.setdefault(t.frame, []).append((tid, t))
    buckets: dict[int, list] = {tid: [] for tid in by_id}
    frames_of: dict[int, list] = {tid: [] for tid in by_id}

    for fi, frame in read_frames(video_source):
        if fi not in frame_boxes:
            continue
        H, W = frame.shape[:2]
        for tid, t in frame_boxes[fi]:
            x1, y1 = max(0, int(t.x)), max(0, int(t.y))
            x2, y2 = min(W, int(t.x + t.w)), min(H, int(t.y + t.h))
            if x2 <= x1 or y2 <= y1:
                continue
            crop = cv2.resize(frame[y1:y2, x1:x2], (size, size))
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            crop = (crop - IMAGENET_MEAN) / IMAGENET_STD
            buckets[tid].append(crop.transpose(2, 0, 1))
            frames_of[tid].append(fi)
    return {tid: (frames_of[tid], np.stack(c)) for tid, c in buckets.items() if c}


# ----------------------- suy luận cảnh báo cho từng mô hình -----------------------
def alarms_rule(tracks, th: RuleThresholds):
    alarms = []
    for seq in _tracks_by_id(tracks).values():
        alarms += merge_alarms(predict_sequence(seq, th))
    return alarms


def _window_flags(frames, scores, seq_len, stride, thresh=0.5):
    """Từ điểm số cửa sổ -> cờ frame (đánh dấu frame cuối mỗi cửa sổ dương)."""
    flags = [False] * (max(frames) + 1) if frames else []
    for (i0, score) in scores:
        if score >= thresh:
            f = frames[min(i0 + seq_len - 1, len(frames) - 1)]
            flags[f] = True
    return flags


def alarms_sequence_model(id_sequences, predict_fn, seq_len=16, stride=8, thresh=0.5):
    """Chạy mô hình chuỗi (M2/M3) trên từng ID bằng cửa sổ trượt -> cảnh báo.

    id_sequences: dict[tid] -> (frames[list[int]], feats np.ndarray[T, ...]).
    predict_fn: callable(batch np.ndarray[N, seq_len, ...]) -> prob np.ndarray[N].
    """
    alarms = []
    for tid, (frames, feats) in id_sequences.items():
        if len(feats) < seq_len:
            continue
        idxs = list(range(0, len(feats) - seq_len + 1, stride))
        batch = np.stack([feats[i:i + seq_len] for i in idxs])
        probs = predict_fn(batch)
        scores = list(zip(idxs, probs))
        flags = _window_flags(frames, scores, seq_len, stride, thresh)
        alarms += merge_alarms(flags)
    return alarms


# ----------------------- chạy một mô hình trên cả tập test -----------------------
def run_rule_on_testset(items, detector: DetectorTracker, th: RuleThresholds | None = None):
    """Đánh giá M1 (rule-based) trên danh sách VideoItem/dict -> EventMetrics gộp."""
    th = th or RuleThresholds()
    per_video, total_sec = [], 0.0
    for it in items:
        path = it["path"] if isinstance(it, dict) else it.path
        tracks = detector.run(path)
        fps = video_fps(it)
        n_frames = (max((t.frame for t in tracks), default=0) + 1)
        seconds = n_frames / fps
        total_sec += seconds
        gt = make_gt_falls(it, n_frames)
        alarms = alarms_rule(tracks, th)
        per_video.append(evaluate_events(alarms, gt, seconds, fps))
    return aggregate_over_dataset(per_video, total_sec)


def _windows_from_id_seq(frames, feats, gt_falls, seq_len, stride):
    """Cắt cửa sổ trượt + gán nhãn theo chồng lấn với khoảng ngã thật."""
    out = []
    for i in range(0, max(1, len(feats) - seq_len + 1), stride):
        win = feats[i:i + seq_len]
        if len(win) < seq_len:
            continue
        wf = frames[i:i + seq_len]
        out.append((np.asarray(win, np.float32), _label_window(wf, gt_falls)))
    return out


def build_pose_samples(items, seq_len=16, stride=8, pose_model="yolo11n-pose.pt"):
    """Tạo mẫu huấn luyện M2: list[(np.ndarray[seq_len, K*2], label)]."""
    samples = []
    for it in items:
        path = it["path"] if isinstance(it, dict) else it.path
        kp_by_id = pose_keypoints(path, pose_model)
        fps = video_fps(it)
        n_frames = max((len(v) for v in kp_by_id.values()), default=0)
        gt = make_gt_falls(it, n_frames)
        for _tid, kp in kp_by_id.items():            # kp: [T, K, 2]
            feats = kp.reshape(len(kp), -1)          # [T, K*2]
            frames = list(range(len(feats)))
            samples += _windows_from_id_seq(frames, feats, gt, seq_len, stride)
    return samples


def build_crop_embedding_samples(items, detector: DetectorTracker, backbone="mobilenet_v2",
                                 seq_len=16, stride=8):
    """Tạo mẫu huấn luyện M3 dạng đặc trưng CNN tiền tính (nhẹ RAM/nhanh hội tụ).

    Trả list[(np.ndarray[seq_len, D], label)]. (Dùng CNNGRU với backbone='precomputed'
    hoặc một MLP+GRU; xem notebook 05 để chọn biến thể.)
    """
    samples = []
    for it in items:
        path = it["path"] if isinstance(it, dict) else it.path
        tracks = detector.run(path)
        fps = video_fps(it)
        n_frames = (max((t.frame for t in tracks), default=0) + 1)
        gt = make_gt_falls(it, n_frames)
        crops_by_id = extract_id_crops(path, tracks)
        for _tid, (frames, crops) in crops_by_id.items():
            emb = cnn_embeddings(crops, backbone)     # [T, D]
            samples += _windows_from_id_seq(frames, emb, gt, seq_len, stride)
    return samples


def make_predict_fn(model, device=None):
    """Bọc model torch thành predict_fn(batch)->prob (numpy)."""
    import torch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    def _fn(batch: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            x = torch.as_tensor(batch, dtype=torch.float32, device=device)
            return torch.sigmoid(model(x)).cpu().numpy()
    return _fn


**Index dữ liệu URFD/Le2i + chia tập theo video**

Quét dữ liệu **URFD/Le2i** thành `VideoItem` (`index_urfd`, `index_le2i`, đọc annotation Le2i), chia tập **theo video/người** chống rò rỉ (`split_by_group`) và tạo các protocol within + cross-dataset (`build_splits`).

In [ ]:
from __future__ import annotations  # chú thích kiểu lười (vd 'str | None'); chạy các cell theo thứ tự từ trên xuống

"""Chia tập theo VIDEO (chống rò rỉ) và giao thức cross-dataset.

Nguyên tắc: không bao giờ chia theo frame — các frame liền kề gần giống nhau
sẽ rò rỉ giữa train/test. Mỗi video (và người thực hiện) chỉ thuộc một tập.
"""
import argparse
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Literal



@dataclass
class VideoItem:
    """Một đơn vị dữ liệu mức video/chuỗi."""
    vid: str                       # id duy nhất
    dataset: str                   # "urfd" | "le2i"
    label: int                     # 1 = có té ngã, 0 = ADL
    path: str
    subject: str | None = None     # người thực hiện (nếu biết) -> chia theo người
    scene: str | None = None       # bối cảnh/góc camera cho TN3
    fall_start: int | None = None  # frame bắt đầu ngã (nếu có nhãn)
    fall_end: int | None = None


_VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}


def index_urfd(root: str | Path) -> list[VideoItem]:
    """Quét thư mục URFD -> danh sách VideoItem (nhãn mức chuỗi).

    URFD gồm 70 chuỗi: fall-01..fall-30 (có té ngã) và adl-01..adl-40 (sinh hoạt
    thường). Mỗi chuỗi thường là một thư mục PNG (ví dụ 'fall-01-cam0-rgb') hoặc
    một file video. Hàm nhận diện nhãn qua tiền tố tên ('fall' -> 1, 'adl' -> 0)
    và camera qua 'cam0'/'cam1'. Dung nạp nhiều cách sắp xếp của các mirror Kaggle.

    URFD không cung cấp bbox người; fall_start/fall_end để None (ở mức sự kiện,
    cả chuỗi fall được coi là một sự kiện ngã — xem datasets.make_gt_falls).
    """
    root = Path(root)
    items: list[VideoItem] = []
    seen: set[str] = set()

    def classify(name: str) -> int | None:
        low = name.lower()
        if low.startswith("fall") or "-fall" in low or "/fall" in low:
            return 1
        if low.startswith("adl") or "-adl" in low or "/adl" in low:
            return 0
        return None

    # 1) Thư mục chuỗi PNG: */<seq>-camX-rgb/ chứa ảnh
    for d in sorted(root.rglob("*")):
        if not d.is_dir():
            continue
        if not any(d.glob("*.png")) and not any(d.glob("*.jpg")):
            continue
        label = classify(d.name)
        if label is None:
            continue
        cam = "cam1" if "cam1" in d.name.lower() else "cam0"
        vid = d.name
        if vid in seen:
            continue
        seen.add(vid)
        items.append(VideoItem(vid=vid, dataset="urfd", label=label, path=str(d),
                               subject=_seq_base(d.name), scene=cam))

    # 2) File video rời (một số mirror đóng gói thành .avi/.mp4)
    for f in sorted(root.rglob("*")):
        if f.suffix.lower() not in _VIDEO_EXTS:
            continue
        label = classify(f.stem)
        if label is None:
            continue
        cam = "cam1" if "cam1" in f.stem.lower() else "cam0"
        vid = f.stem
        if vid in seen:
            continue
        seen.add(vid)
        items.append(VideoItem(vid=vid, dataset="urfd", label=label, path=str(f),
                               subject=_seq_base(f.stem), scene=cam))

    if not items:
        raise FileNotFoundError(
            f"Không tìm thấy chuỗi URFD trong {root}. Kiểm tra lại đường dẫn/'Add Input' trên Kaggle.")
    return items


def _seq_base(name: str) -> str:
    """Lấy id chuỗi gốc (bỏ hậu tố cam/rgb) để chia tập không rò rỉ giữa 2 camera."""
    import re
    m = re.match(r"((?:fall|adl)[-_]?\d+)", name.lower())
    return m.group(1).replace("_", "-") if m else name


def parse_le2i_annotation(ann_path: Path):
    """Đọc annotation Le2i: 2 dòng đầu = frame bắt đầu/kết thúc ngã; các dòng sau =
    'frame,label,x1,y1,x2,y2' (bbox từng frame).

    Trả (fall_start, fall_end, bboxes) với bboxes = {frame: (x1,y1,x2,y2)}.
    Bao dung với biến thể định dạng (dấu phẩy/space/tab, thiếu bbox).
    """
    try:
        lines = [l.strip() for l in ann_path.read_text(errors="ignore").splitlines() if l.strip()]
    except OSError:
        return None, None, {}
    if not lines:
        return None, None, {}

    def ints(s):
        import re
        return [int(float(x)) for x in re.split(r"[,\s]+", s.strip()) if x != ""]

    fall_start = fall_end = None
    body_start = 0
    if len(lines) >= 2 and len(ints(lines[0])) == 1 and len(ints(lines[1])) == 1:
        fall_start, fall_end = ints(lines[0])[0], ints(lines[1])[0]
        body_start = 2

    bboxes: dict[int, tuple] = {}
    for l in lines[body_start:]:
        vals = ints(l)
        if len(vals) >= 6:
            fr, _lab, x1, y1, x2, y2 = vals[:6]
            if x2 > 0 or y2 > 0:
                bboxes[fr] = (x1, y1, x2, y2)
    if fall_start in (0, None) and fall_end in (0, None):
        fall_start = fall_end = None
    return fall_start, fall_end, bboxes


def index_le2i(root: str | Path) -> list[VideoItem]:
    """Quét Le2i (4 bối cảnh) + đọc annotation (frame ngã + bbox).

    Cấu trúc thường gặp: <root>/<Scene>/Videos/*.avi và <root>/<Scene>/Annotation_files/*.txt
    (hoặc annotation đặt cạnh video). Nhãn = 1 nếu annotation có mốc ngã, else 0.
    Scene lấy từ tên thư mục cha (Home/Coffee_room/Office/Lecture_room).
    """
    root = Path(root)
    items: list[VideoItem] = []
    for f in sorted(root.rglob("*")):
        if f.suffix.lower() not in _VIDEO_EXTS:
            continue
        ann = _find_le2i_annotation(f)
        fall_start = fall_end = None
        if ann is not None:
            fall_start, fall_end, _ = parse_le2i_annotation(ann)
        label = 1 if (fall_start is not None and fall_end is not None) else 0
        scene = _le2i_scene(f)
        items.append(VideoItem(vid=f"{scene}-{f.stem}", dataset="le2i", label=label,
                               path=str(f), subject=scene, scene=scene,
                               fall_start=fall_start, fall_end=fall_end))
    if not items:
        raise FileNotFoundError(
            f"Không tìm thấy video Le2i trong {root}. Kiểm tra lại đường dẫn/'Add Input' trên Kaggle.")
    return items


def _le2i_scene(video: Path) -> str:
    known = {"home", "coffee_room", "coffee", "office", "lecture_room", "lecture"}
    for p in video.parents:
        if p.name.lower().replace(" ", "_") in known:
            return p.name
    # thử thư mục cha cấp 1-2
    return video.parent.name


def _find_le2i_annotation(video: Path) -> Path | None:
    stem = video.stem
    cands = [
        video.with_suffix(".txt"),
        video.parent / f"{stem}.txt",
        video.parent.parent / "Annotation_files" / f"{stem}.txt",
        video.parent.parent / "Annotation" / f"{stem}.txt",
    ]
    for c in cands:
        if c.exists():
            return c
    # tìm mờ theo stem
    for c in video.parent.parent.rglob(f"{stem}.txt"):
        return c
    return None


def split_by_group(
    items: list[VideoItem],
    ratio=(0.70, 0.15, 0.15),
    key: Literal["video", "subject"] = "video",
    seed: int = 42,
) -> dict[str, list[str]]:
    """Chia train/val/test theo nhóm (video hoặc người), giữ cân bằng lớp thô."""
    import random
    rng = random.Random(seed)
    # gom theo key để không rò rỉ
    groups: dict[str, list[VideoItem]] = {}
    for it in items:
        gkey = getattr(it, "vid" if key == "video" else key, None) or it.vid
        groups.setdefault(gkey, []).append(it)
    keys = list(groups)
    rng.shuffle(keys)
    n = len(keys)
    n_tr = int(ratio[0] * n)
    n_va = int(ratio[1] * n)
    buckets = {"train": keys[:n_tr], "val": keys[n_tr:n_tr + n_va], "test": keys[n_tr + n_va:]}
    return {s: [v for k in ks for v in [it.vid for it in groups[k]]] for s, ks in buckets.items()}


def build_splits(cfg: dict) -> dict:
    """Tạo cả split within-dataset và 2 protocol cross-dataset."""
    set_seed(cfg["seed"])
    urfd = index_urfd(cfg["paths"]["urfd_root"])
    le2i = index_le2i(cfg["paths"]["le2i_root"])
    ratio = cfg["data"]["split_ratio"]

    out = {
        "within_urfd": split_by_group(urfd, ratio, seed=cfg["seed"]),
        "within_le2i": split_by_group(le2i, ratio, seed=cfg["seed"]),
        # Protocol A: train URFD -> test Le2i ; Protocol B: train Le2i -> test URFD
        "cross_A_train_urfd_test_le2i": {
            "train": [it.vid for it in urfd],
            "test": [it.vid for it in le2i],
        },
        "cross_B_train_le2i_test_urfd": {
            "train": [it.vid for it in le2i],
            "test": [it.vid for it in urfd],
        },
    }
    # lưu cả metadata từng video để dùng lại
    out["_items"] = [asdict(it) for it in (urfd + le2i)]
    save_json(out, Path(cfg["paths"]["splits_dir"]) / "splits.json")
    return out


## Cấu hình & chạy thực nghiệm

In [ ]:
# === Cấu hình (nhúng inline — không cần configs/default.yaml hay src/) ===
cfg = {
    "seed": 42,
    "paths": {
        "urfd_root": "data/urfd",     # TRÊN KAGGLE: /kaggle/input/<slug>/...
        "le2i_root": "data/le2i",     # TRÊN KAGGLE: /kaggle/input/<slug>/...
        "work_dir":  "outputs",       # /kaggle/working khi chạy Kaggle
        "splits_dir": "splits",
    },
    "data": {
        "fps_sample": 5, "img_size": 224, "seq_len": 16,
        "split_ratio": [0.70, 0.15, 0.15], "split_by": "video",
    },
    "detection": {"model": "yolo11n.pt", "conf": 0.25, "person_class": 0},
    "tracking":  {"type": "bytetrack", "track_thresh": 0.5, "match_thresh": 0.8},
    "evaluate":  {"tolerance_sec": 1.0},
    "model2_pose_lstm": {"hidden": 128, "layers": 1, "dropout": 0.3, "lr": 0.001, "epochs": 40},
    "model3_cnn_gru":   {"backbone": "mobilenet_v2", "hidden": 128, "dropout": 0.4, "lr": 0.001, "epochs": 30},
}
set_seed(cfg["seed"])
# TRÊN KAGGLE: trỏ cfg["paths"] tới dataset đã Add Input, ví dụ:
# cfg["paths"]["urfd_root"] = "/kaggle/input/urfd/..."
# cfg["paths"]["le2i_root"] = "/kaggle/input/le2i/..."
print("Kaggle:", on_kaggle())


In [ ]:
from torch.utils.data import DataLoader
items = index_urfd(cfg['paths']['urfd_root'])
# TODO: tách train/val theo splits['within_urfd']; ở đây minh họa:
train_items, val_items = items[:50], items[50:]
tr = build_pose_samples(train_items, cfg['data']['seq_len'])
va = build_pose_samples(val_items, cfg['data']['seq_len'])
ltr = DataLoader(PoseSequenceDataset(tr), batch_size=32, shuffle=True, collate_fn=pad_collate)
lva = DataLoader(PoseSequenceDataset(va), batch_size=32, collate_fn=pad_collate)
in_dim = tr[0][0].shape[-1]
model = PoseLSTM(in_dim=in_dim, hidden=cfg['model2_pose_lstm']['hidden'])
train(model, ltr, lva, cfg['model2_pose_lstm'], 'outputs/model2_best.pt')